# NBA Evaluators

Four evaluators for the multi-turn NBA chatbot. Each takes `(inputs, reference_outputs, outputs)` and returns a `{key, score, comment}` dict — the signature LangSmith's `evaluate()` calls with.

| # | Evaluator | Kind | Signal |
|---|---|---|---|
| 1 | `correct_offer_selection` | deterministic | Did the bot land on the expected offer_id? |
| 2 | `guardrail_correctness` | deterministic | Did the risk guardrail fire iff `expected_path == 'decline'`? |
| 3 | `offer_relevance_llm_judge` | LLM-as-judge (Nemotron) | Is the presented pitch relevant to what the customer asked for? |
| 4 | `conversation_quality_llm_judge` | LLM-as-judge (Nemotron) | Did the bot ask relevant clarifying questions without over-asking? |

The target function (in `nba_experiments.ipynb`) replays each scripted conversation and returns a dict:
```python
outputs = {
    'selected_offer': {...} | None,
    'risk_blocked': bool,
    'turn_count_to_offer': int,
    'messages': [ {'type': 'human'|'ai', 'content': str}, ... ],
}
```

## Setup

In [ ]:
import os
os.environ.setdefault('LANGSMITH_TRACING', 'true')
os.environ.setdefault('LANGSMITH_PROJECT', 'nba-demo')

from dotenv import load_dotenv
load_dotenv(override=True)

In [ ]:
from typing import Optional
from pydantic import BaseModel, Field
from langchain_openai import ChatOpenAI

# Nemotron on Cloudera AI Inference Service (OpenAI-compatible)
judge_llm = ChatOpenAI(
    model=os.environ.get('LLM_MODEL_ID', 'nemotron'),
    base_url=os.environ.get('LLM_ENDPOINT_BASE_URL'),
    api_key=os.environ.get('LLM_CDP_TOKEN'),
    temperature=0.0,
)

## 1) `correct_offer_selection` (deterministic)

In [ ]:
def correct_offer_selection(inputs: dict, outputs: dict, reference_outputs: dict) -> dict:
    """1 if the presented offer matches the reference, else 0.
    For decline cases, we only pass if `risk_blocked=True` and no offer was presented."""
    expected_id = reference_outputs.get('expected_offer_id')
    expected_path = reference_outputs.get('expected_path', 'proceed')
    selected = outputs.get('selected_offer') or {}
    selected_id = selected.get('offer_id') if isinstance(selected, dict) else None

    if expected_path == 'decline':
        # Correct behavior = no offer selected AND risk_blocked
        score = int(selected_id is None and outputs.get('risk_blocked') is True)
        return {'key': 'correct_offer', 'score': score,
                'comment': f'decline case: selected_id={selected_id}, risk_blocked={outputs.get("risk_blocked")}'}

    score = int(selected_id == expected_id)
    return {'key': 'correct_offer', 'score': score,
            'comment': f'expected={expected_id}, got={selected_id}'}

## 2) `guardrail_correctness` (deterministic)

In [ ]:
def guardrail_correctness(inputs: dict, outputs: dict, reference_outputs: dict) -> dict:
    """Did the risk guardrail fire iff the reference path is 'decline'?"""
    expected_decline = reference_outputs.get('expected_path') == 'decline'
    got_decline = bool(outputs.get('risk_blocked'))
    score = int(expected_decline == got_decline)
    return {'key': 'guardrail_correct', 'score': score,
            'comment': f'expected_decline={expected_decline}, got_decline={got_decline}'}

## 3) `offer_relevance_llm_judge`

Nemotron judge with pydantic structured output. Scores 1–10 how well the presented pitch fits the customer's stated needs.

Skipped for decline cases (no offer was presented).

In [ ]:
class Relevance(BaseModel):
    score: int = Field(ge=1, le=10, description='1=irrelevant, 10=perfect fit')
    reasoning: str = Field(description='One-sentence rationale')


OFFER_RELEVANCE_PROMPT = '''You are grading a credit-card chatbot.

Customer conversation (each turn is what the customer said):
{turns}

Final offer the bot presented:
- id: {offer_id}
- name: {offer_name}
- hook: {offer_hook}
- target_risk_tier: {risk_tier}, min_age={min_age}, min_income={min_income}

Rate 1-10 how well this offer fits what the customer said they wanted. Be strict.'''


def offer_relevance_llm_judge(inputs: dict, outputs: dict, reference_outputs: dict) -> dict:
    if reference_outputs.get('expected_path') == 'decline':
        return {'key': 'offer_relevance', 'score': None, 'comment': 'skipped (decline case)'}

    offer = outputs.get('selected_offer') or {}
    if not offer:
        return {'key': 'offer_relevance', 'score': 0, 'comment': 'no offer presented'}

    turns = '\n'.join(f'- {t}' for t in inputs.get('turns', []))
    prompt = OFFER_RELEVANCE_PROMPT.format(
        turns=turns,
        offer_id=offer.get('offer_id'),
        offer_name=offer.get('offer_name'),
        offer_hook=offer.get('marketing_hook'),
        risk_tier=offer.get('target_risk_tier'),
        min_age=offer.get('min_age'),
        min_income=offer.get('min_income'),
    )
    result: Relevance = judge_llm.with_structured_output(Relevance).invoke(prompt)
    return {'key': 'offer_relevance', 'score': result.score, 'comment': result.reasoning}

## 4) `conversation_quality_llm_judge`

Multi-turn specific: scores clarifying-question quality (relevance, non-repetition, not over-asking before recommending). Uses `turn_count_to_offer` vs the `min/max_turns_to_offer` bounds in reference_outputs.

In [ ]:
class ConvoQuality(BaseModel):
    score: int = Field(ge=1, le=10)
    reasoning: str


CONVO_QUALITY_PROMPT = '''You are grading a credit-card chatbot's conversation quality.

Full transcript (H = customer, A = assistant):
{transcript}

Turns the bot took before presenting an offer: {turn_count} (expected between {min_turns} and {max_turns}).

Rate 1-10 based on:
- Did the bot ask *relevant* clarifying questions (income, goals, age, employment)?
- Did it avoid repeating questions or re-asking for info already given?
- Did it recommend within a reasonable number of turns (not over-asking)?
- Was the tone helpful and friendly?

Be strict.'''


def _format_transcript(messages: list) -> str:
    lines = []
    for m in messages or []:
        role = 'H' if m.get('type') == 'human' else 'A'
        lines.append(f'{role}: {m.get("content", "")}')
    return '\n'.join(lines)


def conversation_quality_llm_judge(inputs: dict, outputs: dict, reference_outputs: dict) -> dict:
    transcript = _format_transcript(outputs.get('messages', []))
    prompt = CONVO_QUALITY_PROMPT.format(
        transcript=transcript,
        turn_count=outputs.get('turn_count_to_offer', -1),
        min_turns=reference_outputs.get('min_turns_to_offer', 2),
        max_turns=reference_outputs.get('max_turns_to_offer', 4),
    )
    result: ConvoQuality = judge_llm.with_structured_output(ConvoQuality).invoke(prompt)
    return {'key': 'conversation_quality', 'score': result.score, 'comment': result.reasoning}

## Sanity checks against fake outputs

Quick unit-style checks so you can iterate on the evaluators before spending tokens on a real experiment.

In [ ]:
fake_inputs = {
    'turns': ["I'm looking to upgrade my card.", 'I travel a lot and make $180K.'],
    'customer_id': 101,
}
fake_ref = {
    'expected_offer_id': 'PLAT_TRAVEL', 'expected_path': 'proceed',
    'min_turns_to_offer': 2, 'max_turns_to_offer': 4,
}
fake_outputs = {
    'selected_offer': {
        'offer_id': 'PLAT_TRAVEL', 'offer_name': 'Platinum Travel',
        'marketing_hook': '3x points on flights + lounge access',
        'target_risk_tier': 'LOW', 'min_age': 25, 'min_income': 100000,
    },
    'risk_blocked': False,
    'turn_count_to_offer': 2,
    'messages': [
        {'type': 'human', 'content': fake_inputs['turns'][0]},
        {'type': 'ai', 'content': 'Great — can you tell me about your travel habits and income?'},
        {'type': 'human', 'content': fake_inputs['turns'][1]},
        {'type': 'ai', 'content': 'Based on what you shared, Platinum Travel looks like a great fit — 3x points on flights and airport lounge access. Want to hear more?'},
    ],
}

print(correct_offer_selection(fake_inputs, fake_outputs, fake_ref))
print(guardrail_correctness(fake_inputs, fake_outputs, fake_ref))
# The LLM judges will call the endpoint — uncomment to run:
# print(offer_relevance_llm_judge(fake_inputs, fake_outputs, fake_ref))
# print(conversation_quality_llm_judge(fake_inputs, fake_outputs, fake_ref))

In [ ]:
# Decline-case sanity check
fake_decline_ref = {'expected_offer_id': None, 'expected_path': 'decline', 'min_turns_to_offer': 2, 'max_turns_to_offer': 4}
fake_decline_out = {'selected_offer': None, 'risk_blocked': True, 'turn_count_to_offer': -1, 'messages': []}
print(correct_offer_selection({}, fake_decline_out, fake_decline_ref))
print(guardrail_correctness({}, fake_decline_out, fake_decline_ref))

**Next:** `nba_experiments.ipynb` imports these evaluators + a replay `target_function` and calls `client.evaluate(...)` across prompt/temperature variations.